# Initialize

In [25]:
model=f'model/model-test.232000'
host='35.226.127.248'
host_dir='/home/hmatsuya/dlcobra/usi/bin'
username='hmatsuya'

drive='/mnt/c'
spawner_path=drive + '/Users/hmats/Downloads/Shogidokoro/Engine/ssh_gcp.bat'


## Convert

In [3]:
import os
os.getcwd()
# 'C:\\Users\\hmats\\workspace\\DeepLearningShogi'

'C:\\Users\\hmats\\workspace\\DeepLearningShogi'

In [4]:
activation='C:\\Anaconda3\\Scripts\\activate.bat'
onnx_path=f'x64/Release/{os.path.basename(model)}.onnx'
print('model    :', model)
print('onnx_path:', onnx_path)

! C:\\Anaconda3\\Scripts\\activate.bat & python dlshogi\\convert_model_to_onnx.py {model} {onnx_path} --network resnet10_swish


model    : model/model-test.232000
onnx_path: x64/Release/model-test.232000.onnx
graph(%input1 : Float(1:5022, 62:81, 9:9, 9:1, requires_grad=0, device=cuda:0),
      %input2 : Float(1:4617, 57:81, 9:9, 9:1, requires_grad=0, device=cuda:0),
      %l1_1_1.weight : Float(192:558, 62:9, 3:3, 3:1, requires_grad=1, device=cuda:0),
      %l1_1_2.weight : Float(192:62, 62:1, 1:1, 1:1, requires_grad=1, device=cuda:0),
      %l1_2.weight : Float(192:57, 57:1, 1:1, 1:1, requires_grad=1, device=cuda:0),
      %l22.weight : Float(27:192, 192:1, 1:1, 1:1, requires_grad=1, device=cuda:0),
      %l22_2.bias : Float(2187:1, requires_grad=1, device=cuda:0),
      %l23_v.weight : Float(256:2187, 2187:1, requires_grad=1, device=cuda:0),
      %l23_v.bias : Float(256:1, requires_grad=1, device=cuda:0),
      %l24_v.weight : Float(1:256, 256:1, requires_grad=1, device=cuda:0),
      %l24_v.bias : Float(1:1, requires_grad=1, device=cuda:0),
      %norm1.weight : Float(192:1, requires_grad=1, device=cuda:0),

## Upload

In [29]:
from paramiko import Transport, SFTPClient, RSAKey, WarningPolicy

pk=RSAKey.from_private_key(open('/Users/hmats/.ssh/id_rsa'))

transport = Transport(sock=(host, 22))
transport.connect(username=username, pkey=pk)
connection = SFTPClient.from_transport(transport)
# connection.set_missing_host_key_policy(WarningPolicy)
connection.put(
    localpath=onnx_path,
    remotepath=f'{host_dir}/{os.path.basename(onnx_path)}',
#     callback=self.uploading_info,
    confirm=True
)

<SFTPAttributes: [ size=29379537 uid=1001 gid=1002 mode=0o100664 atime=1619087003 mtime=1619087075 ]>

## Setup spawner script

In [2]:
with open(spawner_path, mode='w') as f:
    f.write(f'@echo off\nssh hmatsuya@{host} "cd {host_dir} && ./usi"')

# Match

In [2]:
import os
os.getcwd()

'/mnt/c/Users/hmats/workspace/DeepLearningShogi/notebook'

In [3]:
import sys
sys.path.insert(1, '../utils')

In [4]:
#!pip install cshogi

In [5]:
from importlib import reload  
import matches2
reload(matches2)

<module 'matches2' from '../utils/matches2.py'>

In [13]:
max_gpu = 4
model_option =  f"DNN_Model:{os.path.basename(model)}.onnx," + ','.join([f'DNN_Model{i}:{os.path.basename(model)}.onnx' for i in range(2, max_gpu + 1)])
model_option

'DNN_Model:model-test.232000.onnx,DNN_Model2:model-test.232000.onnx,DNN_Model3:model-test.232000.onnx,DNN_Model4:model-test.232000.onnx'

In [14]:
threads = 7
thread_option =  f"UCT_Threads:{threads}," + ','.join([f'UCT_Threads{i}:{threads}' for i in range(2, max_gpu + 1)])
thread_option

'UCT_Threads:7,UCT_Threads2:7,UCT_Threads3:7,UCT_Threads4:7'

In [15]:
batch_size = 256
batch_option =  f"DNN_Batch_Size:{batch_size}," + ','.join([f'DNN_Batch_Size{i}:{batch_size}' for i in range(2, max_gpu + 1)])
batch_option

'DNN_Batch_Size:256,DNN_Batch_Size2:256,DNN_Batch_Size3:256,DNN_Batch_Size4:256'

In [ ]:
arglist = [
#     "x64/Release/dlshogi_tensorrt.exe",
#     drive + "/Users/hmats/workspace/YaneuraOu6/NNUE/YaneuraOu_NNUE-evallearn-g++-zen2.exe",
#     "../x64/Release/dlshogi_tensorrt.exe",
#     "/Users/hmats/workspace/YaneuraOu6/NNUE/YaneuraOu_NNUE-evallearn-g++-zen2.exe",
    drive + "/Users/hmats/Downloads/Shogidokoro/Engine/ssh_gcp.bat",
    drive + "/Users/hmats/Downloads/Shogidokoro/Engine/ssh_gcp.bat",
#     "--options1", "DNN_Model:model.onnx",
#     "--options1", "USI_Ponder:false,Threads:64,USI_Hash:4096,NetworkDelay:0,NetworkDelay2:0",
    "--options1", f"{model_option},{thread_option},{batch_option},Softmax_Temperature:174",
    "--options2", f"{model_option},{thread_option},{batch_option},Softmax_Temperature:348",
#     "--options2", "USI_Ponder:false,Treads:16,USI_Hash:4096,NetworkDelay:0,NetworkDelay2:0",
    "--games", "300",
    "--byoyomi", "1000",
    "--max_turn", "256",
    "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（010手目、評価値±100以内、492局面）.sfen",
#     "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（020手目、評価値±100以内、748局面）.sfen",
#     "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（030手目、評価値±100以内、547局面）.sfen",
#    "initial_positions": "",
#    "kifu_dir": "",
#    "log": "",
#    "debug": "",
]
match_args = matches2.parse_arguments(arglist)
matches2.matches(match_args)

OXOO-OOOOO 0.889
OO-OOXOOOO 0.889
OO-XOX--OO 0.84
OO-OOOOXOO 0.853
XXOXOOOO-X 0.791
-OOXOOOOOO 0.808
-OX--O-XOX 0.776
OXOOOXOOOO 0.779
XOOOOOOO-X 0.779
-OOOOXXO-O 0.776
OOOOOO--

In [9]:
arglist = [
#     "x64/Release/dlshogi_tensorrt.exe",
#     drive + "/Users/hmats/workspace/YaneuraOu6/NNUE/YaneuraOu_NNUE-evallearn-g++-zen2.exe",
#     "../x64/Release/dlshogi_tensorrt.exe",
#     "/Users/hmats/workspace/YaneuraOu6/NNUE/YaneuraOu_NNUE-evallearn-g++-zen2.exe",
    drive + "/Users/hmats/Downloads/Shogidokoro/Engine/ssh_gcp.bat",
    drive + "/Users/hmats/Downloads/Shogidokoro/Engine/ssh_gcp.bat",
#     "--options1", "DNN_Model:model.onnx",
#     "--options1", "USI_Ponder:false,Threads:64,USI_Hash:4096,NetworkDelay:0,NetworkDelay2:0",
    "--options1", f"{model_option},{thread_option},{batch_option},Softmax_Temperature:174",
    "--options2", f"{model_option},{thread_option},{batch_option},Softmax_Temperature:200",
#     "--options2", "USI_Ponder:false,Treads:16,USI_Hash:4096,NetworkDelay:0,NetworkDelay2:0",
    "--games", "300",
    "--byoyomi", "1000",
    "--max_turn", "256",
    "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（010手目、評価値±100以内、492局面）.sfen",
#     "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（020手目、評価値±100以内、748局面）.sfen",
#     "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（030手目、評価値±100以内、547局面）.sfen",
#    "initial_positions": "",
#    "kifu_dir": "",
#    "log": "",
#    "debug": "",
]
match_args = matches2.parse_arguments(arglist)
matches2.matches(match_args)

OOXXOO--XO 0.625
OOXXO-OOXX 0.588
OOXOXXOXXO 0.556
X-OO

KeyboardInterrupt: 

In [16]:
threads = 7
thread_option2 =  f"UCT_Threads:{threads}," + ','.join([f'UCT_Threads{i}:{threads}' for i in range(2, max_gpu + 1)])
thread_option2

'UCT_Threads:7,UCT_Threads2:7,UCT_Threads3:7,UCT_Threads4:7'

In [17]:
batch_size = 128
batch_option2 =  f"DNN_Batch_Size:{batch_size}," + ','.join([f'DNN_Batch_Size{i}:{batch_size}' for i in range(2, max_gpu + 1)])
batch_option2

'DNN_Batch_Size:128,DNN_Batch_Size2:128,DNN_Batch_Size3:128,DNN_Batch_Size4:128'

In [18]:
arglist = [
#     "x64/Release/dlshogi_tensorrt.exe",
#     drive + "/Users/hmats/workspace/YaneuraOu6/NNUE/YaneuraOu_NNUE-evallearn-g++-zen2.exe",
#     "../x64/Release/dlshogi_tensorrt.exe",
#     "/Users/hmats/workspace/YaneuraOu6/NNUE/YaneuraOu_NNUE-evallearn-g++-zen2.exe",
    drive + "/Users/hmats/Downloads/Shogidokoro/Engine/ssh_gcp.bat",
    drive + "/Users/hmats/Downloads/Shogidokoro/Engine/ssh_gcp.bat",
#     "--options1", "DNN_Model:model.onnx",
#     "--options1", "USI_Ponder:false,Threads:64,USI_Hash:4096,NetworkDelay:0,NetworkDelay2:0",
    "--options1", f"{model_option},{thread_option},{batch_option},Softmax_Temperature:174",
    "--options2", f"{model_option},{thread_option2},{batch_option2},Softmax_Temperature:174",
#     "--options2", "USI_Ponder:false,Treads:16,USI_Hash:4096,NetworkDelay:0,NetworkDelay2:0",
    "--games", "300",
    "--byoyomi", "1000",
    "--max_turn", "256",
#     "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（010手目、評価値±100以内、492局面）.sfen",
    "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（020手目、評価値±100以内、748局面）.sfen",
#     "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（030手目、評価値±100以内、547局面）.sfen",
#    "initial_positions": "",
#    "kifu_dir": "",
#    "log": "",
#    "debug": "",
]
match_args = matches2.parse_arguments(arglist)
matches2.matches(match_args)

XXXXXX-X-X 0.0
XOXOXXXXOX 0.167
OXOXOOOXOX 0.321
XXOXOXOXXO 0.342
XXOXXO

KeyboardInterrupt: 

In [20]:
threads = 7
thread_option =  f"UCT_Threads:{threads}," + ','.join([f'UCT_Threads{i}:{threads}' for i in range(2, max_gpu + 1)])
thread_option

'UCT_Threads:7,UCT_Threads2:7,UCT_Threads3:7,UCT_Threads4:7'

In [21]:
batch_size = 128
batch_option =  f"DNN_Batch_Size:{batch_size}," + ','.join([f'DNN_Batch_Size{i}:{batch_size}' for i in range(2, max_gpu + 1)])
batch_option

'DNN_Batch_Size:128,DNN_Batch_Size2:128,DNN_Batch_Size3:128,DNN_Batch_Size4:128'

In [22]:
threads = 7
thread_option2 =  f"UCT_Threads:{threads}," + ','.join([f'UCT_Threads{i}:{threads}' for i in range(2, max_gpu + 1)])
thread_option2

'UCT_Threads:7,UCT_Threads2:7,UCT_Threads3:7,UCT_Threads4:7'

In [23]:
batch_size = 96
batch_option2 =  f"DNN_Batch_Size:{batch_size}," + ','.join([f'DNN_Batch_Size{i}:{batch_size}' for i in range(2, max_gpu + 1)])
batch_option2

'DNN_Batch_Size:96,DNN_Batch_Size2:96,DNN_Batch_Size3:96,DNN_Batch_Size4:96'

In [26]:
arglist = [
#     "x64/Release/dlshogi_tensorrt.exe",
#     drive + "/Users/hmats/workspace/YaneuraOu6/NNUE/YaneuraOu_NNUE-evallearn-g++-zen2.exe",
#     "../x64/Release/dlshogi_tensorrt.exe",
#     "/Users/hmats/workspace/YaneuraOu6/NNUE/YaneuraOu_NNUE-evallearn-g++-zen2.exe",
    drive + "/Users/hmats/Downloads/Shogidokoro/Engine/ssh_gcp.bat",
    drive + "/Users/hmats/Downloads/Shogidokoro/Engine/ssh_gcp.bat",
#     "--options1", "DNN_Model:model.onnx",
#     "--options1", "USI_Ponder:false,Threads:64,USI_Hash:4096,NetworkDelay:0,NetworkDelay2:0",
    "--options1", f"{model_option},{thread_option},{batch_option},Softmax_Temperature:174",
    "--options2", f"{model_option},{thread_option2},{batch_option2},Softmax_Temperature:174",
#     "--options2", "USI_Ponder:false,Treads:16,USI_Hash:4096,NetworkDelay:0,NetworkDelay2:0",
    "--games", "300",
    "--byoyomi", "1000",
    "--max_turn", "256",
#     "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（010手目、評価値±100以内、492局面）.sfen",
    "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（020手目、評価値±100以内、748局面）.sfen",
#     "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（030手目、評価値±100以内、547局面）.sfen",
#    "initial_positions": "",
#    "kifu_dir": "",
#    "log": "",
#    "debug": "",
]
match_args = matches2.parse_arguments(arglist)
matches2.matches(match_args)

O-OXO-XO-O 0.714
XXOXOXOXOX 0.529
OOOX--OO

KeyboardInterrupt: 

In [27]:
threads = 7
thread_option =  f"UCT_Threads:{threads}," + ','.join([f'UCT_Threads{i}:{threads}' for i in range(2, max_gpu + 1)])
thread_option

'UCT_Threads:7,UCT_Threads2:7,UCT_Threads3:7,UCT_Threads4:7'

In [28]:
batch_size = 128
batch_option =  f"DNN_Batch_Size:{batch_size}," + ','.join([f'DNN_Batch_Size{i}:{batch_size}' for i in range(2, max_gpu + 1)])
batch_option

'DNN_Batch_Size:128,DNN_Batch_Size2:128,DNN_Batch_Size3:128,DNN_Batch_Size4:128'

In [29]:
threads = 7
thread_option2 =  f"UCT_Threads:{threads}," + ','.join([f'UCT_Threads{i}:{threads}' for i in range(2, max_gpu + 1)])
thread_option2

'UCT_Threads:7,UCT_Threads2:7,UCT_Threads3:7,UCT_Threads4:7'

In [30]:
batch_size = 160
batch_option2 =  f"DNN_Batch_Size:{batch_size}," + ','.join([f'DNN_Batch_Size{i}:{batch_size}' for i in range(2, max_gpu + 1)])
batch_option2

'DNN_Batch_Size:160,DNN_Batch_Size2:160,DNN_Batch_Size3:160,DNN_Batch_Size4:160'

In [31]:
arglist = [
#     "x64/Release/dlshogi_tensorrt.exe",
#     drive + "/Users/hmats/workspace/YaneuraOu6/NNUE/YaneuraOu_NNUE-evallearn-g++-zen2.exe",
#     "../x64/Release/dlshogi_tensorrt.exe",
#     "/Users/hmats/workspace/YaneuraOu6/NNUE/YaneuraOu_NNUE-evallearn-g++-zen2.exe",
    drive + "/Users/hmats/Downloads/Shogidokoro/Engine/ssh_gcp.bat",
    drive + "/Users/hmats/Downloads/Shogidokoro/Engine/ssh_gcp.bat",
#     "--options1", "DNN_Model:model.onnx",
#     "--options1", "USI_Ponder:false,Threads:64,USI_Hash:4096,NetworkDelay:0,NetworkDelay2:0",
    "--options1", f"{model_option},{thread_option},{batch_option},Softmax_Temperature:174",
    "--options2", f"{model_option},{thread_option2},{batch_option2},Softmax_Temperature:174",
#     "--options2", "USI_Ponder:false,Treads:16,USI_Hash:4096,NetworkDelay:0,NetworkDelay2:0",
    "--games", "30",
    "--byoyomi", "1000",
    "--max_turn", "256",
#     "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（010手目、評価値±100以内、492局面）.sfen",
    "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（020手目、評価値±100以内、748局面）.sfen",
#     "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（030手目、評価値±100以内、547局面）.sfen",
#    "initial_positions": "",
#    "kifu_dir": "",
#    "log": "",
#    "debug": "",
]
match_args = matches2.parse_arguments(arglist)
matches2.matches(match_args)

XXOXXOOOXO 0.5
XOOOOXOXXO 0.55
OO-OO

KeyboardInterrupt: 

In [32]:
threads = 3
thread_option =  f"UCT_Threads:{threads}," + ','.join([f'UCT_Threads{i}:{threads}' for i in range(2, max_gpu + 1)])
thread_option

'UCT_Threads:3,UCT_Threads2:3,UCT_Threads3:3,UCT_Threads4:3'

In [33]:
batch_size = 128
batch_option =  f"DNN_Batch_Size:{batch_size}," + ','.join([f'DNN_Batch_Size{i}:{batch_size}' for i in range(2, max_gpu + 1)])
batch_option

'DNN_Batch_Size:128,DNN_Batch_Size2:128,DNN_Batch_Size3:128,DNN_Batch_Size4:128'

In [34]:
threads = 7
thread_option2 =  f"UCT_Threads:{threads}," + ','.join([f'UCT_Threads{i}:{threads}' for i in range(2, max_gpu + 1)])
thread_option2

'UCT_Threads:7,UCT_Threads2:7,UCT_Threads3:7,UCT_Threads4:7'

In [35]:
batch_size = 128
batch_option2 =  f"DNN_Batch_Size:{batch_size}," + ','.join([f'DNN_Batch_Size{i}:{batch_size}' for i in range(2, max_gpu + 1)])
batch_option2

'DNN_Batch_Size:128,DNN_Batch_Size2:128,DNN_Batch_Size3:128,DNN_Batch_Size4:128'

In [36]:
arglist = [
#     "x64/Release/dlshogi_tensorrt.exe",
#     drive + "/Users/hmats/workspace/YaneuraOu6/NNUE/YaneuraOu_NNUE-evallearn-g++-zen2.exe",
#     "../x64/Release/dlshogi_tensorrt.exe",
#     "/Users/hmats/workspace/YaneuraOu6/NNUE/YaneuraOu_NNUE-evallearn-g++-zen2.exe",
    drive + "/Users/hmats/Downloads/Shogidokoro/Engine/ssh_gcp.bat",
    drive + "/Users/hmats/Downloads/Shogidokoro/Engine/ssh_gcp.bat",
#     "--options1", "DNN_Model:model.onnx",
#     "--options1", "USI_Ponder:false,Threads:64,USI_Hash:4096,NetworkDelay:0,NetworkDelay2:0",
    "--options1", f"{model_option},{thread_option},{batch_option},Softmax_Temperature:174",
    "--options2", f"{model_option},{thread_option2},{batch_option2},Softmax_Temperature:174",
#     "--options2", "USI_Ponder:false,Treads:16,USI_Hash:4096,NetworkDelay:0,NetworkDelay2:0",
    "--games", "30",
    "--byoyomi", "1000",
    "--max_turn", "256",
#     "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（010手目、評価値±100以内、492局面）.sfen",
    "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（020手目、評価値±100以内、748局面）.sfen",
#     "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（030手目、評価値±100以内、547局面）.sfen",
#    "initial_positions": "",
#    "kifu_dir": "",
#    "log": "",
#    "debug": "",
]
match_args = matches2.parse_arguments(arglist)
matches2.matches(match_args)

OXOOO-OXXX 0.556
XXXXOXOOOO 0.526
XX-OXXOXOX 0.464


0.4642857142857143

In [37]:
threads = 7
thread_option =  f"UCT_Threads:{threads}," + ','.join([f'UCT_Threads{i}:{threads}' for i in range(2, max_gpu + 1)])
thread_option

'UCT_Threads:7,UCT_Threads2:7,UCT_Threads3:7,UCT_Threads4:7'

In [38]:
batch_size = 128
batch_option =  f"DNN_Batch_Size:{batch_size}," + ','.join([f'DNN_Batch_Size{i}:{batch_size}' for i in range(2, max_gpu + 1)])
batch_option

'DNN_Batch_Size:128,DNN_Batch_Size2:128,DNN_Batch_Size3:128,DNN_Batch_Size4:128'

In [39]:
arglist = [
#     "x64/Release/dlshogi_tensorrt.exe",
#     drive + "/Users/hmats/workspace/YaneuraOu6/NNUE/YaneuraOu_NNUE-evallearn-g++-zen2.exe",
#     "../x64/Release/dlshogi_tensorrt.exe",
#     "/Users/hmats/workspace/YaneuraOu6/NNUE/YaneuraOu_NNUE-evallearn-g++-zen2.exe",
    drive + "/Users/hmats/Downloads/Shogidokoro/Engine/ssh_gcp.bat",
    drive + "/Users/hmats/Downloads/Shogidokoro/Engine/ssh_gcp.bat",
#     "--options1", "DNN_Model:model.onnx",
#     "--options1", "USI_Ponder:false,Threads:64,USI_Hash:4096,NetworkDelay:0,NetworkDelay2:0",
    "--options1", f"{model_option},{thread_option},{batch_option},Softmax_Temperature:174",
    "--options2", f"{model_option},{thread_option},{batch_option},Softmax_Temperature:150",
#     "--options2", "USI_Ponder:false,Treads:16,USI_Hash:4096,NetworkDelay:0,NetworkDelay2:0",
    "--games", "30",
    "--byoyomi", "1000",
    "--max_turn", "256",
#     "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（010手目、評価値±100以内、492局面）.sfen",
    "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（020手目、評価値±100以内、748局面）.sfen",
#     "--initial_positions", "../../ShogiGokakuKyokumen/2020年/floodgate2020（R4000以上）/互角局面集/互角局面集（030手目、評価値±100以内、547局面）.sfen",
#    "initial_positions": "",
#    "kifu_dir": "",
#    "log": "",
#    "debug": "",
]
match_args = matches2.parse_arguments(arglist)
matches2.matches(match_args)

OX-OO-XX-X 0.429
XOXOOXXXXO 0.412
XOXOOOO-OO 0.538


0.5384615384615384